# 数理変換タクティクス AI トレーニング
このノートブックでゲームのAIをトレーニングし、TensorFlow.js形式でエクスポートします。

## 手順
1. 依存ライブラリのインストール
2. ゲームシミュレーションの定義
3. トレーニングデータ生成（自己対戦）
4. ニューラルネットワークのトレーニング
5. TF.js形式でエクスポート
6. ZIPをダウンロードしてGitHubにアップロード

In [ ]:
# ライブラリインストール
!pip install tensorflowjs -q
print('インストール完了！')

In [ ]:
import numpy as np
import tensorflow as tf
import tensorflowjs as tfjs
import random
import math

print(f'TensorFlow: {tf.__version__}')
print(f'TensorFlow.js converter: {tfjs.__version__}')

## ゲームシミュレーター
ゲームのルールをPythonで再現します。

In [ ]:
class MathCardGame:
    """
    数理変換タクティクス シミュレーター
    アクションカテゴリ:
      0 = パス（ターン終了）
      1 = 約数強奪攻撃
      2 = 足し算（2枚合成）
    """
    
    def __init__(self, config=None):
        self.config = config or {
            'initHandCount': 5,
            'initMaxNum': 10,
            'drawCount': 2,
            'drawMaxNum': 20,
            'handLimitNum': 150,
            'winScore': 100,
            'maxTurns': 10,
        }
    
    def reset(self):
        cfg = self.config
        self.hands = {
            'P1': [random.randint(1, cfg['initMaxNum']) for _ in range(cfg['initHandCount'])],
            'P2': [random.randint(1, cfg['initMaxNum']) for _ in range(cfg['initHandCount'])],
        }
        self.scores = {'P1': 0, 'P2': 0}
        self.turn_count = 1
        self.abs_turn = 0
        self.current_player = 'P1'
        self.done = False
        self.winner = None
    
    def opponent(self, player_id):
        return 'P2' if player_id == 'P1' else 'P1'
    
    def get_state_vector(self, player_id):
        """30次元の状態ベクトルに変換（cpu-ai.jsと同じ形式）"""
        vector = [0.0] * 30
        my_hand = self.hands[player_id]
        enemy_id = self.opponent(player_id)
        enemy_hand = self.hands[enemy_id]
        
        for i, card in enumerate(my_hand[:10]):
            vector[i] = card / 150.0
        vector[10] = self.scores[player_id] / 100.0
        
        for i, card in enumerate(enemy_hand[:10]):
            vector[11 + i] = card / 150.0
        vector[21] = self.scores[enemy_id] / 100.0
        
        vector[22] = self.turn_count / 10.0
        return vector
    
    def best_attack(self, player_id):
        """最も多くのカードを奪える攻撃カードを返す"""
        enemy_id = self.opponent(player_id)
        best = None
        best_gain = 0
        for i, atk in enumerate(self.hands[player_id]):
            if atk == 1:
                continue
            gain = sum(n for n in self.hands[enemy_id] if n % atk == 0)
            if gain > best_gain:
                best_gain = gain
                best = (i, atk, gain)
        return best
    
    def best_add(self, player_id):
        """最大の足し算結果を出すペアを返す"""
        hand = self.hands[player_id]
        limit = self.config['handLimitNum']
        best = None
        best_val = -1
        for i in range(len(hand)):
            for j in range(len(hand)):
                if i == j:
                    continue
                res = hand[i] + hand[j]
                if res <= limit and res > best_val:
                    best_val = res
                    best = (i, j, res)
        return best
    
    def get_optimal_action(self, player_id):
        """
        ルールベースの最適行動
        → これを使ってトレーニングデータを生成する
        """
        atk = self.best_attack(player_id)
        add = self.best_add(player_id)
        
        my_score = self.scores[player_id]
        enemy_score = self.scores[self.opponent(player_id)]
        win_score = self.config['winScore']
        
        # 攻撃で勝てるなら即攻撃
        if atk and self.turn_count > 1:
            if my_score + atk[2] >= win_score:
                return 1
        
        # 敵の手が多く、攻撃が有効なら攻撃
        if atk and self.turn_count > 1:
            enemy_hand_size = len(self.hands[self.opponent(player_id)])
            if atk[2] > 5 or enemy_hand_size >= 7:
                return 1
        
        # 足し算で大きなカードを作れるなら合成
        if add and add[2] >= 15:
            return 2
        
        # 攻撃がわずかでもあるなら攻撃
        if atk and self.turn_count > 1:
            return 1
        
        # とにかく合成
        if add:
            return 2
        
        return 0  # パス
    
    def execute_action(self, player_id, action_category):
        """行動を実行してターン終了"""
        reward = 0
        enemy_id = self.opponent(player_id)
        
        if action_category == 1:
            atk = self.best_attack(player_id)
            if atk:
                i, atk_num, gain = atk
                self.hands[enemy_id] = [n for n in self.hands[enemy_id] if n % atk_num != 0]
                self.hands[player_id].pop(i)
                self.scores[player_id] += gain
                reward = gain
                if self.scores[player_id] >= self.config['winScore']:
                    self.done = True
                    self.winner = player_id
                    reward += 50
        
        elif action_category == 2:
            add = self.best_add(player_id)
            if add:
                i, j, res = add
                new_hand = [c for k, c in enumerate(self.hands[player_id]) if k != i and k != j]
                new_hand.append(res)
                # ターン開始時のドロー
                cfg = self.config
                for _ in range(cfg['drawCount']):
                    new_hand.append(random.randint(1, cfg['drawMaxNum']))
                self.hands[player_id] = new_hand
                reward = res / 50.0
        
        # ターン終了処理
        self.current_player = enemy_id
        self.abs_turn += 1
        if self.current_player == 'P1':
            self.turn_count += 1
            if self.turn_count > self.config['maxTurns']:
                self.done = True
                self.winner = max(self.scores, key=lambda p: self.scores[p])
        
        return reward

print('ゲームシミュレーター準備完了！')

## トレーニングデータ生成
ルールベースAI同士を自己対戦させて、状態→最適行動 のペアを大量に集めます。

In [ ]:
NUM_EPISODES = 20000  # エピソード数（増やすと精度向上）

X_data = []  # 状態ベクトル
y_data = []  # 最適行動ラベル

game = MathCardGame()
wins = {'P1': 0, 'P2': 0, 'draw': 0}

for episode in range(NUM_EPISODES):
    game.reset()
    
    while not game.done:
        player = game.current_player
        state_vec = game.get_state_vector(player)
        action = game.get_optimal_action(player)
        
        X_data.append(state_vec)
        y_data.append(action)
        
        game.execute_action(player, action)
    
    if game.winner:
        wins[game.winner] = wins.get(game.winner, 0) + 1
    else:
        wins['draw'] = wins.get('draw', 0) + 1
    
    if (episode + 1) % 5000 == 0:
        print(f'エピソード {episode + 1}/{NUM_EPISODES} 完了... データ数: {len(X_data)}')

X_data = np.array(X_data, dtype=np.float32)
y_data = np.array(y_data, dtype=np.int32)

print(f'\n生成完了！')
print(f'総データ数: {len(X_data)}')
print(f'勝率 P1:{wins["P1"]}, P2:{wins.get("P2",0)}, draw:{wins.get("draw",0)}')

unique, counts = np.unique(y_data, return_counts=True)
for u, c in zip(unique, counts):
    label = ['パス', '攻撃', '足し算'][u]
    print(f'  行動{u}({label}): {c}件 ({c/len(y_data)*100:.1f}%)')

## ニューラルネットワーク トレーニング

In [ ]:
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# データを訓練/検証に分割
X_train, X_val, y_train, y_val = train_test_split(
    X_data, y_data, test_size=0.1, random_state=42
)

# ラベルをone-hot形式に変換
y_train_cat = to_categorical(y_train, num_classes=3)
y_val_cat = to_categorical(y_val, num_classes=3)

# モデル定義（cpu-ai.jsと互換: 入力30次元 → 出力3クラス）
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(30,), name='dense_1'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu', name='dense_2'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu', name='dense_3'),
    tf.keras.layers.Dense(3, activation='softmax', name='output'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# コールバック
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
]

history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=50,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)

val_acc = max(history.history['val_accuracy'])
print(f'\n最高検証精度: {val_acc:.4f} ({val_acc*100:.1f}%)')

In [ ]:
# 学習曲線をプロット
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], label='訓練精度')
ax1.plot(history.history['val_accuracy'], label='検証精度')
ax1.set_title('精度')
ax1.legend()
ax1.set_xlabel('Epoch')

ax2.plot(history.history['loss'], label='訓練損失')
ax2.plot(history.history['val_loss'], label='検証損失')
ax2.set_title('損失')
ax2.legend()
ax2.set_xlabel('Epoch')

plt.tight_layout()
plt.show()

## TensorFlow.js 形式でエクスポート
ゲームがブラウザで使えるように変換します。

In [ ]:
import os
import shutil

OUTPUT_DIR = 'tfjs_model'

# 既存フォルダを削除してクリーンにする
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

# TF.js形式で保存
tfjs.converters.save_keras_model(model, OUTPUT_DIR)

print(f'エクスポート完了！ファイル一覧:')
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f} ({size/1024:.1f} KB)')

In [ ]:
# ZIPに圧縮してダウンロード
shutil.make_archive('tfjs_model', 'zip', 'tfjs_model')

from google.colab import files
files.download('tfjs_model.zip')

print('ダウンロード開始！')
print()
print('次のステップ:')
print('1. ZIPを解凍して中のファイルを取り出す')
print('2. GitHubリポジトリの tfjs_model/ フォルダに入れる')
print('3. git add tfjs_model/ && git commit -m "Add trained AI model" && git push')
print('4. ゲームをリロードすると新しいAIが使われます！')

## AIの動作テスト（オプション）
トレーニングしたAIがどんな行動を取るかテストします。

In [ ]:
action_names = ['パス（ターン終了）', '約数強奪攻撃', '足し算合成']

# テスト局面1: 攻撃チャンスあり
test_game = MathCardGame()
test_game.reset()
test_game.hands['P1'] = [6, 3, 10, 7, 2]   # 6は敵の12,18の約数
test_game.hands['P2'] = [12, 18, 5, 9, 4]
test_game.scores = {'P1': 20, 'P2': 30}
test_game.turn_count = 3

vec = np.array([test_game.get_state_vector('P1')], dtype=np.float32)
pred = model.predict(vec, verbose=0)[0]
chosen = np.argmax(pred)
optimal = test_game.get_optimal_action('P1')

print('=== テスト1: 攻撃チャンスあり ===')
print(f'自分の手: {test_game.hands["P1"]}')
print(f'相手の手: {test_game.hands["P2"]}')
print(f'スコア: 自分{test_game.scores["P1"]}点 vs 相手{test_game.scores["P2"]}点')
print(f'AI選択: {action_names[chosen]} (確率: パス{pred[0]:.2f}, 攻撃{pred[1]:.2f}, 足し算{pred[2]:.2f})')
print(f'最適行動: {action_names[optimal]}')
print(f'正解: {"✅" if chosen == optimal else "❌"}')
print()

# テスト局面2: 合成が有利
test_game.hands['P1'] = [5, 8, 3, 2, 7]
test_game.hands['P2'] = [1, 1, 1, 1, 1]   # 敵手が全部1なので攻撃無意味
test_game.turn_count = 2

vec = np.array([test_game.get_state_vector('P1')], dtype=np.float32)
pred = model.predict(vec, verbose=0)[0]
chosen = np.argmax(pred)
optimal = test_game.get_optimal_action('P1')

print('=== テスト2: 合成が有利 ===')
print(f'自分の手: {test_game.hands["P1"]}')
print(f'相手の手: {test_game.hands["P2"]} (全部1→攻撃無意味)')
print(f'AI選択: {action_names[chosen]} (確率: パス{pred[0]:.2f}, 攻撃{pred[1]:.2f}, 足し算{pred[2]:.2f})')
print(f'最適行動: {action_names[optimal]}')
print(f'正解: {"✅" if chosen == optimal else "❌"}')